# <center style="font-family: consolas; font-size: 32px; font-weight: bold;">  Hands-On LangChain for LLM Applications Development: Output Parsing </center>

# <center style="font-family: consolas; font-size: 25px; font-weight: bold;">  (OpenRouter + Google Colab + LangChain 1.x / Pydantic Edition) </center>
***

When developing a complex application with a Language Model (LLM), it's common to specify the desired output format, such as JSON, and designate particular keys for organizing the data.

Historically, LangChain offered `ResponseSchema` + `StructuredOutputParser` to coax an LLM into producing parseable JSON via prompt instructions. **As of modern LangChain (1.x), this approach is deprecated / removed from the mainstream API.** The current, officially recommended way to get structured output is:

1. Define your desired output shape as a **Pydantic** `BaseModel`.
2. Bind it to the chat model with **`llm.with_structured_output(YourModel)`**.
3. Call `.invoke(...)` and get back a validated Python object directly — no manual prompt-engineering of format instructions, and no manual string parsing.

This notebook walks through both the *old problem* (why a plain LLM string isn't usable as data) and the *modern solution* (Pydantic + `with_structured_output`), running on **OpenRouter** so you can swap in any model (OpenAI, Anthropic, Google, Meta, etc.) with a single API key.


# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 0. Setting Up Working Environment with OpenRouter on Colab </b></div>

Let's install the required libraries first, then load our OpenRouter API key and set up the client.


In [ ]:
# Install required libraries (Colab)
!pip install -q openai langchain langchain-core langchain-openai pydantic


### OpenRouter API Key

Create an account at [openrouter.ai](https://openrouter.ai/keys) and get an API key.

On Colab, store the key in **Secrets** (the 🔑 icon on the left sidebar) under the name `OPENROUTER_API_KEY`. If the secret isn't found, you'll be prompted to enter it manually.


In [ ]:
import os

try:
    # If you're running this on Google Colab
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
except Exception:
    OPENROUTER_API_KEY = None

if not OPENROUTER_API_KEY:
    import getpass
    OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API key: ")

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY


In [ ]:
from openai import OpenAI

# OpenAI SDK client, but pointed at the OpenRouter endpoint
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# Pick any model available on OpenRouter that supports tool/function calling
# (with_structured_output relies on this under the hood):
# https://openrouter.ai/models?supported_parameters=tools
# e.g. "openai/gpt-4o-mini", "openai/gpt-4.1-mini", "anthropic/claude-3.5-sonnet"
llm_model = "openai/gpt-4o-mini"


# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 1. The Problem: Raw LLM Output Is Just a String </b></div>


Let's start with an example to clarify the output parsing concept. Here's an example of how you'd *like* the output formatted, as a real Python dictionary — this is our target shape:


In [ ]:
x = {
  "gift": False,
  "delivery_days": 5,
  "price_value": ["pretty affordable!"]
}


In [ ]:
x.get("price_value")


Here is an example customer review, plus a naive prompt template that simply *asks* the LLM to output JSON as text.


In [ ]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""


So here's how you can wrap this in LangChain. First, we import the chat prompt template and build a `ChatPromptTemplate` from the review template above.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(review_template)
print(prompt_template)


Let's create the messages, call the OpenRouter endpoint, and print out the response.


In [ ]:
from langchain_openai import ChatOpenAI

messages = prompt_template.format_messages(text=customer_review)

chat = ChatOpenAI(
    temperature=0.0,
    model=llm_model,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
)

response = chat.invoke(messages)
print(response.content)


It looks right — `gift` looks true, `delivery_days` looks like 2, `price_value` looks reasonable. But if we check the type of the response, it's just a `str`. It *looks* like JSON, but it isn't a dictionary — it's one long string (often even wrapped in ```` ```json ... ``` ```` markdown fences), which is fragile to parse and easy to break.


In [ ]:
type(response.content)


In [ ]:
# This fails: response.content is a string, not a dict.
# response.content.get('gift')   # -> AttributeError: 'str' object has no attribute 'get'

response.content[5]


This is exactly the problem structured output is meant to solve. Historically, LangChain's answer was `ResponseSchema` + `StructuredOutputParser`, which auto-generated formatting instructions to stuff into the prompt, then parsed the resulting string back into a dict.

**That approach is now deprecated.** In modern LangChain (1.x), the recommended solution is `with_structured_output()` combined with a **Pydantic** model — no manual format-instruction prompt engineering, and no manual string parsing. Let's do it the modern way.


# <div style="box-shadow: rgba(0, 0, 0, 0.16) 0px 1px 4px inset, rgb(51, 51, 51) 0px 0px 0px 3px inset; padding:20px; font-size:32px; font-family: consolas; text-align:center; display:fill; border-radius:15px;  color:rgb(34, 34, 34);"> <b> 2. The Modern Way: Pydantic + with_structured_output() </b></div>


Step 1 — define the exact shape you want as a **Pydantic model**. Each field's type is enforced automatically (bool, int, list of strings...), and the `description` on each `Field` tells the LLM what to extract, replacing the old `ResponseSchema` descriptions.


In [ ]:
from pydantic import BaseModel, Field
from typing import List


class ReviewAnalysis(BaseModel):
    gift: bool = Field(
        description="True if the item was purchased as a gift for someone else, otherwise False."
    )
    delivery_days: int = Field(
        description="Number of days it took for the product to arrive. Return -1 if unknown."
    )
    price_value: List[str] = Field(
        description="Sentences from the text that discuss the product's price or value."
    )


Step 2 — bind this schema directly to the chat model with `with_structured_output(...)`. This tells the model (via tool/function calling under the hood) to always return data matching `ReviewAnalysis`, validated automatically.


In [ ]:
structured_llm = chat.with_structured_output(ReviewAnalysis)


Step 3 — just `.invoke()` it with plain text. No prompt template, no format instructions needed — the schema itself *is* the instruction.


In [ ]:
result = structured_llm.invoke(customer_review)
result


Notice the type: this is a real `ReviewAnalysis` Pydantic object, not a string. You can access fields directly with dot notation — no parsing, no `.get()` guesswork.


In [ ]:
print(type(result))
print(result.gift)
print(result.delivery_days)
print(result.price_value)


If you need a plain Python dictionary (e.g. to send as JSON, store in a database, etc.), Pydantic gives you that in one call:


In [ ]:
result.model_dump()


### Bonus: swapping providers is a one-line change

Because `with_structured_output()` is a LangChain abstraction, the same `ReviewAnalysis` schema works with **any** chat model that supports tool calling — you're not locked into OpenAI. For example, if you're connected to Google's Gemini via `langchain_google_genai` instead of going through OpenRouter:

```python
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
structured_llm = llm.with_structured_output(ReviewAnalysis)

result = structured_llm.invoke(customer_review)
print(result)
```

Same schema, same `.invoke()` call, different backend — that's the whole point of the abstraction.


### Summary of changes made in this version

- **Removed** the deprecated `ResponseSchema` / `StructuredOutputParser` workflow entirely (it no longer has a direct equivalent in modern LangChain).
- **Replaced** it with the officially recommended pattern: a **Pydantic `BaseModel`** describing the desired shape, bound to the LLM via **`llm.with_structured_output(YourModel)`**.
- No more manually injecting `{format_instructions}` into the prompt template, and no more manually calling `output_parser.parse(response.content)` — the model now returns a validated object directly from `.invoke()`.
- Kept the OpenRouter + Colab setup (Secrets-based API key, OpenAI-compatible `base_url`) so you can point `llm_model` at any tool-calling-capable model (OpenAI, Anthropic, etc.) with one line.
- Added a note on how the exact same schema/pattern ports to other providers (e.g. Gemini via `langchain_google_genai`) with only the model constructor changing.

# <div style="box-shadow: rgba(240, 46, 170, 0.4) -5px 5px inset, rgba(240, 46, 170, 0.3) -10px 10px inset, rgba(240, 46, 170, 0.2) -15px 15px inset, rgba(240, 46, 170, 0.1) -20px 20px inset, rgba(240, 46, 170, 0.05) -25px 25px inset; padding:20px; font-size:30px; font-family: consolas; display:fill; border-radius:15px; color: rgba(240, 46, 170, 0.7)"> <b> ༼⁠ ⁠つ⁠ ⁠◕⁠‿⁠◕⁠ ⁠༽⁠つ Thank You!</b></div>
